# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I build a leakage-safe vector for the decline label: static page properties, the earlier
`*_prev_30d` traffic window (knowable before the outcome), and safe categoricals. Heavy-tailed
traffic counts get a `log1p`. Because keyword and length fields go missing along `content_type`
lines, I add `has_`-flags first so the missingness is an explicit signal, then fill — never a
blind `fillna(0)` that would smuggle content type into every zero.

In [1]:
import pandas as pd, numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

num_safe = ["content_age_days", "days_since_last_update", "word_count", "char_count",
            "search_volume", "competition", "cpc",
            "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
cat_safe = ["content_type", "main_intent", "competition_level"]

X = df[num_safe + cat_safe].copy()
X["has_keyword_data"] = df["search_volume"].notna().astype(int)
X["has_word_count"] = df["word_count"].notna().astype(int)
for col in ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]:
    X["log_" + col] = np.log1p(X[col].fillna(0))
X[num_safe] = X[num_safe].fillna(0)
X[cat_safe] = X[cat_safe].fillna("unknown")

print("feature vector:", X.shape)
print("numeric   :", [c for c in X.columns if c not in cat_safe])
print("categorical:", cat_safe)


feature vector: (30000, 18)
numeric   : ['content_age_days', 'days_since_last_update', 'word_count', 'char_count', 'search_volume', 'competition', 'cpc', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'has_keyword_data', 'has_word_count', 'log_impressions_prev_30d', 'log_clicks_prev_30d', 'log_sessions_prev_30d']
categorical: ['content_type', 'main_intent', 'competition_level']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE prediction.*

| Feature | Meaning | Missing handling | Known before the outcome? |
|---|---|---|---|
| `content_age_days`, `days_since_last_update` | age / staleness of the page | complete | yes, static |
| `word_count`, `char_count` | length | `has_word_count` flag, then 0 | yes, static |
| `search_volume`, `competition`, `cpc` | keyword context | `has_keyword_data` flag, then 0 | yes, static |
| `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` | traffic in days 31–60 back | complete; `log1p` | yes, strictly before the last-30 label window |
| `content_type`, `main_intent`, `competition_level` | categoricals | filled `"unknown"` | yes, static |

The missingness table below is why the `has_`-flags exist rather than a silent fill.

In [2]:
miss = df[num_safe + cat_safe].isna().mean().round(3).sort_values(ascending=False)
print("missing share per raw feature:")
print(miss.to_string())

print()
print("keyword missingness by content_type:")
by_type = df.assign(no_kw=df["search_volume"].isna()).groupby("content_type")["no_kw"].mean().round(3)
print(by_type.to_string())


missing share per raw feature:
word_count                0.257
char_count                0.257
competition_level         0.087
competition               0.082
search_volume             0.082
cpc                       0.082
main_intent               0.079
days_since_last_update    0.000
content_age_days          0.000
clicks_prev_30d           0.000
impressions_prev_30d      0.000
content_type              0.000
sessions_prev_30d         0.000

keyword missingness by content_type:
content_type
comparison article    0.000
feedly article        1.000
keyword article       0.014


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

The test that matters: train once with honest features, then add a suspect and watch the score.
The split is grouped by `client_id` so a client's pages never sit in both train and test, and the
base rate sits next to every number. A jump toward a perfect AUC is a confession, not a win.

- **`trend_pct`** is the label source, so adding it should collapse the score toward 1.0.
- **`impressions_last_30d`** overlaps the label's window, so it should lift the score unfairly too.
- Honest features alone should land only modestly above the base rate — which is the real signal.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

y = df["is_declining_label"].values
groups = df["client_id"].values
base = X.select_dtypes("number")

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42).split(base, y, groups))

def auc_with(extra_cols):
    feats = base.copy()
    for col in extra_cols:
        feats[col] = df[col].fillna(0).values
    model = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
    model.fit(feats.iloc[tr], y[tr])
    return roc_auc_score(y[te], model.predict_proba(feats.iloc[te])[:, 1])

print(f"base rate (declining)          = {y.mean():.3f}")
print(f"honest features only     AUC   = {auc_with([]):.3f}")
print(f"+ trend_pct (label source) AUC = {auc_with(['trend_pct']):.3f}")
print(f"+ impressions_last_30d     AUC = {auc_with(['impressions_last_30d']):.3f}")


base rate (declining)          = 0.542
honest features only     AUC   = 0.623
+ trend_pct (label source) AUC = 1.000


+ impressions_last_30d     AUC = 0.665


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Each exclusion is a specific leak or privacy risk, not caution for its own sake.

In [4]:
excluded = {
    "trend_direction / trend_pct / is_declining_label": "label source; the target is computed from them",
    "impressions_last_30d, *_90d, ctr, engagement_rate": "90d / last-30 windows overlap the label window",
    "content_id / client_id": "pseudonymous identity; grouping and splits only, never learned",
    "provider_used / model_used": "content-generation metadata, not a prediction-time signal",
    "avg_position == 0": "means no position data, not a real rank",
    "health_score (warehouse)": "product-decision flag; using it learns the old rule, not the world",
    "raw URLs / query text (warehouse)": "private, unsafe to publish",
}
for field, why in excluded.items():
    print(f"- {field}: {why}")


- trend_direction / trend_pct / is_declining_label: label source; the target is computed from them
- impressions_last_30d, *_90d, ctr, engagement_rate: 90d / last-30 windows overlap the label window
- content_id / client_id: pseudonymous identity; grouping and splits only, never learned
- provider_used / model_used: content-generation metadata, not a prediction-time signal
- avg_position == 0: means no position data, not a real rank
- health_score (warehouse): product-decision flag; using it learns the old rule, not the world
- raw URLs / query text (warehouse): private, unsafe to publish


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.